# IFN680 Project 6: The Sharpness Quest - DigitClassifier

**Group 4** - Karan Rooprai (n12498122), Nhu Hieu Nguyen (n12194778)

This notebook trains the **standalone digit classifier** that `main_report.ipynb` uses to evaluate generated digits. It is an instrument, not one of the models being compared. Lecture 9 uses the same method: on p42 a classifier trained on real digits reaches 99% or more on them and 98% on a conditional GAN's samples, and p87 lists classifying generated data among the ways to measure it.

It serves two purposes in `main_report.ipynb`:

- **class consistency**: a sample drawn under class 3 should be recognised as a 3;
- **a feature space**: its 128-unit layer describes a digit by 128 numbers, and comparing the average and spread of those numbers for real and generated digits gives a Frechet distance, the idea behind FID, in a space built for digits.

It trains for a fixed five epochs on all 60,000 training images and nothing is tuned, so no development split is needed. It never makes a prediction on the test split. It writes `DigitClassifier.pth`.

**Sections:** 1 Setup, 2 The data, 3 The model, 4 Training.

## 1 Setup

The libraries, the device and the seed, as in the two cVAE notebooks.

In [1]:
# The Week 8 tutorial's stack. json carries results from the training notebooks to
# main_report.ipynb, and make_grid lays out digit grids the way the tutorial's figures do.
import json
import os
import platform
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader, TensorDataset
from torchvision.utils import make_grid

# The brief asks for the IFN680 GPU environment. The same code runs on a CPU when no GPU is
# visible, so the notebook runs anywhere and every tensor is placed with .to(device).
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"device      : {device}")
print(f"python      : {platform.python_version()}")
print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"numpy       : {np.__version__}")

# One seed for everything: the data split, the initial weights and every random draw.
SEED = 0
np.random.seed(SEED)
_ = torch.manual_seed(SEED)

device      : cuda:0
python      : 3.12.14
torch       : 2.13.0+cu126
torchvision : 0.28.0+cu126
numpy       : 2.5.3


## 2 The data

`mnist_custom.pt` holds four tensors: 60,000 training images with their labels, and 10,000 test
images with theirs. The cell below loads it. Because `main_report.ipynb` needs the test split and
nothing else, the cell also writes that split to its own file, `mnist_custom_test.pt`, so the
archive does not have to carry the whole 112 MB dataset.

In [2]:
# mnist_custom.pt is the dataset supplied with the brief: MNIST resized to 20 x 20 and
# inverted. weights_only=True loads the tensors and refuses to run any code stored in the file.
data = torch.load("mnist_custom.pt", weights_only=True)
train_images, train_labels = data["train_images"], data["train_labels"]

# main_report.ipynb needs the test split and nothing else, so it is written to its own file
# here. No training notebook scores it or makes a prediction on it.
if not os.path.exists("mnist_custom_test.pt"):
    test_split = {key: data[key] for key in ("test_images", "test_labels")}
    torch.save(test_split, "mnist_custom_test.pt")
print(f"loaded mnist_custom.pt: {', '.join(data.keys())}")

loaded mnist_custom.pt: train_images, train_labels, test_images, test_labels


### 2.1 What the file holds

The brief calls the data only "the modified MNIST dataset", so its properties are measured before
anything is trained on it. Two of them matter later. The images are **20 x 20**, not the
tutorial's 28 x 28, which is why the tutorial's model needs a change (`cVAE_Baseline.ipynb`, section 3). And the
background is **white** (pixel value 1.0) with dark strokes, the reverse of the tutorial's MNIST.

In [3]:
# What the file actually holds, measured before anything is trained on it.
border = torch.cat([train_images[..., 0, :], train_images[..., -1, :],
                    train_images[..., :, 0], train_images[..., :, -1]], dim=-1)
print(f"train_images : {tuple(train_images.shape)}, {train_images.dtype}")
print(f"train_labels : {tuple(train_labels.shape)}, {train_labels.dtype}")
print(f"pixel range  : {train_images.min():.3f} to {train_images.max():.3f}")
print(f"class counts : {torch.bincount(train_labels, minlength=10).tolist()}")
# The background is white (1.0) and the strokes dark: the reverse of the tutorial's MNIST.
print(f"pixels exactly 1.0 : {(train_images == 1).float().mean():.3f}")
print(f"mean border pixel  : {border.mean():.4f}")

train_images : (60000, 1, 20, 20), torch.float32
train_labels : (60000,), torch.int64
pixel range  : 0.000 to 1.000
class counts : [5923, 6742, 5958, 6131, 5842, 5421, 5918, 6265, 5851, 5949]
pixels exactly 1.0 : 0.701
mean border pixel  : 0.9993


### 2.2 The training images

The same seeded order as the cVAE notebooks. The classifier trains on all of it, the fit and development splits together, because nothing about it is chosen on held-out data.

In [4]:
# The development split: a seeded random draw of N_DEV training images, held out to judge
# convergence and to choose settings. The final models retrain on all N_TOTAL images.
N_TOTAL, N_DEV = 60000, 10000
split_generator = torch.Generator().manual_seed(SEED)
order = torch.randperm(len(train_images), generator=split_generator)[:N_TOTAL]
dev_index, fit_index = order[:N_DEV], order[N_DEV:]
print(f"fit split         : {len(fit_index):,} images")
print(f"development split : {len(dev_index):,} images")
dev_counts = torch.bincount(train_labels[dev_index], minlength=10).tolist()
print(f"development class counts : {dev_counts}")
print(f"refit set (fit + development) : {len(order):,} images")

fit split         : 50,000 images
development split : 10,000 images
development class counts : [953, 1167, 1018, 1066, 967, 921, 920, 1045, 931, 1012]
refit set (fit + development) : 60,000 images


## 3 The model

Two convolution blocks, each followed by **max pooling**, which keeps the largest value in each 2 x 2 square and so halves the map (20 to 10 to 5), then a 128-unit layer and the 10 class scores. `features()` returns the 128-unit layer.

| Call | What you give it | What you get back |
|---|---|---|
| `nn.MaxPool2d(2)` | maps (batch, c, n, n) | (batch, c, n/2, n/2), the maximum of each 2 x 2 square |
| `F.cross_entropy(logits, labels)` | class scores and true classes | the mean negative log-probability of the true class |

In [5]:
# The evaluation classifier, in the style of lecture 9 p42's: two convolution blocks and a
# 128-unit layer. features() returns that layer, the space the Frechet distances use.
class DigitClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32x10x10
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 64x5x5
            nn.Flatten(),
            nn.Linear(64 * 5 * 5, 128), nn.ReLU(),
        )
        self.output_layer = nn.Linear(128, num_classes)

    def features(self, x):
        return self.feature_extractor(x)

    def forward(self, x):
        return self.output_layer(self.features(x))

## 4 Training

Adam with a learning rate of 0.001, batches of 256, five epochs. Then a check that the loss ended far below the ln 10 = 2.303 of guessing, and the weights are saved from the CPU.

In [6]:
# Checkpoints are saved from the CPU, so they load on a machine without a GPU.
def cpu_state(module):
    return {name: tensor.detach().cpu().clone() for name, tensor in module.state_dict().items()}

In [7]:
# Five fixed epochs on every training image, with no model selection: the classifier is a
# measuring instrument, so nothing about it is tuned and no development split is needed.
NUM_CLASSES, CLASSIFIER_EPOCHS = 10, 5
_ = torch.manual_seed(SEED)
classifier = DigitClassifier(NUM_CLASSES).to(device)
optimizer = optim.Adam(classifier.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(train_images[order], train_labels[order]), batch_size=256,
                    shuffle=True, generator=torch.Generator().manual_seed(SEED))
print(f"classifier parameters: {sum(p.numel() for p in classifier.parameters()):,}")

classifier parameters: 225,034


The loop: score a batch, measure the cross-entropy, step.

In [8]:
# A standard supervised loop: cross-entropy between the logits and the true class.
for epoch in range(CLASSIFIER_EPOCHS):
    classifier.train()
    loss_sum, correct, count = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = classifier(images)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(images)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += len(images)
    print(f"epoch {epoch + 1}: training loss {loss_sum / count:.4f}, "
          f"training accuracy {correct / count:.4f}")
# Chance is a loss of ln 10 = 2.303; a working classifier ends far below it.
assert loss_sum / count < np.log(NUM_CLASSES) / 2, "the classifier did not learn"

epoch 1: training loss 0.5239, training accuracy 0.8525


epoch 2: training loss 0.1008, training accuracy 0.9695


epoch 3: training loss 0.0729, training accuracy 0.9777


epoch 4: training loss 0.0571, training accuracy 0.9823


epoch 5: training loss 0.0485, training accuracy 0.9847


Saving the weights.

In [9]:
# A plain state_dict saved from the CPU; main_report.ipynb rebuilds the class and loads it.
torch.save(cpu_state(classifier), "DigitClassifier.pth")
print("saved DigitClassifier.pth")

saved DigitClassifier.pth
